<a href="https://colab.research.google.com/github/GaurabSingh012/AMR_Btech_Project/blob/main/BtechProjectCV.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!unzip -q "/content/drive/MyDrive/yolo_ball_dataset.zip" -d /content/

In [ ]:
%%writefile /content/yolo_ball_dataset/data.yaml
path: /content/yolo_ball_dataset
train: images/train
val: images/val
test: images/test

names:
  0: blue
  1: green
  2: pink
  3: red
  4: yellow

Overwriting /content/yolo_ball_dataset/data.yaml


In [ ]:
!pip install -q ultralytics

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.9/46.9 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 36.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 96.5/96.5 kB 11.3 MB/s eta 0:00:00


In [ ]:
from ultralytics import YOLO

model = YOLO("yolov8n.pt")

results = model.train(
    data="/content/yolo_ball_dataset/data.yaml",
    epochs=100,
    imgsz=640,
    batch=16,
    device=0,
    workers=2,
    project="/content/runs",
    name="ball-colour-yolov8n"
)

Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings.
Ultralytics 8.4.174 🚀 Python-3.13.15 torch-2.11.0+cu130 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/yolo_ball_dataset/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fract

In [ ]:
from google.colab import files

files.download(
    "/content/runs/ball-colour-yolov8n/weights/best.pt"
)

In [ ]:
from google.colab import files

uploaded = files.upload()
image_path = next(iter(uploaded))

print("Testing image:", image_path)

In [ ]:
from pathlib import Path
from ultralytics import YOLO
from IPython.display import display
from PIL import Image
import matplotlib.pyplot as plt

# Find your trained model
preferred_path = Path(
    "/content/runs/ball-colour-yolov8n/weights/best.pt"
)

if preferred_path.exists():
    model_path = preferred_path
else:
    candidates = list(Path("/content").rglob("best.pt"))

    if not candidates:
        print("Trained best.pt not found.")
        print("Available model files:")
        for p in Path("/content").rglob("*.pt"):
            print(p)
        raise FileNotFoundError(
            "Locate your trained best.pt before testing."
        )

    model_path = candidates[0]

print("Model:", model_path)

# Load your trained model
model = YOLO(str(model_path))

# Test the uploaded image
image_path = "/content/test.jpeg"

results = model.predict(
    source=image_path,
    imgsz=640,
    conf=0.25,
    save=True,
    project="/content/predictions",
    name="test",
    exist_ok=True
)

# Display the image with bounding boxes
annotated = results[0].plot()

display(
    Image.fromarray(annotated[:, :, ::-1])
)

# Print detections
boxes = results[0].boxes

if boxes is None or len(boxes) == 0:
    print("No balls detected.")
else:
    print("\nDetected balls:")

    for box in boxes:
        class_id = int(box.cls.item())
        confidence = float(box.conf.item())
        name = model.names[class_id]

        print(
            f"Class: {name} | "
            f"Confidence: {confidence:.2%}"
        )

print("\nAnnotated image saved to:")
print("/content/predictions/test/")

In [ ]:
import os
import io
import logging
import warnings
from pathlib import Path
from contextlib import redirect_stdout, redirect_stderr
from IPython.display import display, Image

os.environ["YOLO_VERBOSE"] = "False"

from ultralytics import YOLO
from ultralytics.utils import LOGGER

LOGGER.setLevel(logging.ERROR)
warnings.filterwarnings("ignore")

model_path = Path(
    "/content/runs/ball-colour-yolov8n/weights/best.pt"
)
data_path = Path(
    "/content/yolo_ball_dataset/data.yaml"
)

if not model_path.exists():
    raise FileNotFoundError(f"Model not found: {model_path}")

if not data_path.exists():
    raise FileNotFoundError(f"Dataset not found: {data_path}")

# Load and evaluate the trained model
with redirect_stdout(io.StringIO()), redirect_stderr(io.StringIO()):
    model = YOLO(str(model_path))

    metrics = model.val(
        data=str(data_path),
        split="test",
        imgsz=640,
        device=0,
        plots=True,
        verbose=False,
        project="/content/evaluation",
        name="ball-test",
        exist_ok=True
    )

# Collect the evaluation results
precision = metrics.box.mp * 100
recall = metrics.box.mr * 100
map50 = metrics.box.map50 * 100
map5095 = metrics.box.map * 100

inference_ms = metrics.speed["inference"]
fps = 1000 / inference_ms if inference_ms > 0 else 0

test_labels = Path(
    "/content/yolo_ball_dataset/labels/test"
)
test_images = len(list(test_labels.glob("*.txt")))

# Prepare a clean report
report = [
    "YOLOv8 BALL COLOUR DETECTION",
    "TEST EVALUATION RESULTS",
    "",
    f"Model: YOLOv8n",
    f"Test images: {test_images}",
    f"Image size: 640 x 640",
    f"Classes: {', '.join(str(v) for v in model.names.values())}",
    "",
    "OVERALL METRICS",
    f"Precision: {precision:.2f}%",
    f"Recall: {recall:.2f}%",
    f"mAP@50: {map50:.2f}%",
    f"mAP@50-95: {map5095:.2f}%",
    "",
    "PER-CLASS mAP@50-95"
]

for class_id, score in enumerate(metrics.box.maps):
    report.append(
        f"{model.names[class_id]}: {score * 100:.2f}%"
    )

report.extend([
    "",
    "INFERENCE PERFORMANCE",
    f"Preprocessing: {metrics.speed['preprocess']:.2f} ms/image",
    f"Inference: {inference_ms:.2f} ms/image",
    f"Postprocessing: {metrics.speed['postprocess']:.2f} ms/image",
    f"Approximate inference-only FPS: {fps:.2f}",
    "",
    "Note: FPS is measured on the Colab GPU, not the deployment PC."
])

report_text = "\n".join(report)

print(report_text)

# Save the text report
report_path = Path("/content/evaluation_metrics.txt")
report_path.write_text(report_text, encoding="utf-8")

print("\nReport saved to:", report_path)

# Display training and evaluation graphs
graph_folders = [
    Path("/content/runs/ball-colour-yolov8n"),
    Path("/content/evaluation/ball-test")
]

graph_names = [
    "results.png",
    "confusion_matrix.png",
    "confusion_matrix_normalized.png",
    "BoxPR_curve.png",
    "BoxF1_curve.png",
    "BoxP_curve.png",
    "BoxR_curve.png",
    "PR_curve.png",
    "F1_curve.png",
    "P_curve.png",
    "R_curve.png"
]

print("\nTRAINING AND EVALUATION GRAPHS")

shown = set()

for folder in graph_folders:
    for name in graph_names:
        graph_path = folder / name

        if graph_path.exists() and graph_path not in shown:
            print(f"\n{name}")
            display(Image(filename=str(graph_path)))
            shown.add(graph_path)

if not shown:
    print("No graph images found in the result folders.")